# 04. Делегирование: агент как команда

## Что агент пока не умеет

После `03` у нас есть OpenClaw-подобный контур, но один агент начинает перегружаться контекстом и ролями.

> Subagent — это не ещё одна модель ради красивой схемы. Это способ отделить контекст, специализацию и ответственность.


![Subagents delegation](../visuals/openclaw_path/09_subagents_delegation.svg)


## Три причины появления subagents

### Изоляция контекста

Researcher может прочитать десятки файлов, а основной агент получает компактный отчёт.

### Специализация

У subagent отдельный prompt и ограниченная ответственность.

### Разделение ответственности

Один ищет факты, другой критикует выводы.


## Роли

```text
repo-researcher
Цель: найти факты и релевантные участки кода
Доступ: read/search filesystem
Не должен: менять файлы или делать неподтверждённые выводы
```

```text
reviewer
Цель: проверить предложенное решение
Доступ: filesystem, tests, diff
Не должен: самостоятельно переписывать всё решение
```


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

from deepagents import create_deep_agent
from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')

DEFAULT_MODEL = 'openrouter:tencent/hy3:free'

def model_name() -> str:
    return os.getenv('OPENCLAW_MODEL', DEFAULT_MODEL)

def write_text(path: str, content: str) -> Path:
    target = REPO_ROOT / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return target

def write_json(path: str, payload: dict) -> Path:
    target = REPO_ROOT / path
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + '\n')
    return target


In [ ]:
ENTRYPOINT = "\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom connectors.jenkins import JENKINS_TOOLS\nfrom connectors.vk import VK_TOOLS\n\nload_dotenv()\nDEFAULT_MODEL = \"openrouter:tencent/hy3:free\"\n\n\ndef _backend():\n    from deepagents.backends import FilesystemBackend\n    return FilesystemBackend(root_dir=Path(os.getenv(\"OPENCLAW_WORKSPACE\", \".\")).resolve(), virtual_mode=True)\n\nTOOLS = [*JENKINS_TOOLS, *VK_TOOLS]\nSUBAGENTS = [\n    {\n        \"name\": \"repo-researcher\",\n        \"description\": \"Find facts and relevant code locations. Use for repository and integration analysis.\",\n        \"system_prompt\": \"You find facts in the repository. Read/search files, cite paths, and avoid edits or unsupported claims.\",\n    },\n    {\n        \"name\": \"reviewer\",\n        \"description\": \"Review proposed findings and plans for correctness, risk, and missing tests.\",\n        \"system_prompt\": \"You verify claims and plans. Focus on bugs, security, tests, and operational risk. Do not rewrite the whole solution.\",\n    },\n]\nBASE_PROMPT = \"\"\"\\\nYou are OpenClaw at stage 04. For repository integration analysis, delegate to repo-researcher first,\nthen ask reviewer to check the findings before finalizing. This is hierarchical delegation, not a broad multi-agent network.\n\"\"\"\n\nagent = create_deep_agent(\n    model=os.getenv(\"OPENCLAW_MODEL\", DEFAULT_MODEL),\n    tools=TOOLS,\n    system_prompt=BASE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(),\n)\n"
CONFIG = {"dependencies": ["."], "graphs": {"openclaw_04": "./agents/openclaw_04_subagents.py:agent"}, "env": ".env"}
print(write_text('agents/openclaw_04_subagents.py', ENTRYPOINT).relative_to(REPO_ROOT))
print(write_json('langgraph.openclaw_path.json', CONFIG).relative_to(REPO_ROOT))


## Проверка в LangGraph Studio

### Запрос

```text
Проанализируй интеграции Jenkins и VK. Найди три наиболее серьёзных источника нестабильности и попроси reviewer проверить выводы.
```

### Ожидаемое поведение

1. Основной агент делегирует исследование `repo-researcher`.
2. Researcher читает файлы connectors/scripts.
3. Reviewer проверяет выводы.
4. Основной агент возвращает компактный список рисков.

### Что изменилось относительно предыдущего этапа

Появилась иерархическая специализация: контекст исследования отделён от финального ответа.

### Текущее ограничение

Делегирование помогает анализировать и проверять, но пока система не замыкает полный цикл изменения проекта.
